# Parkin: comparison of two MAVE experiments (Google Colab version)

**Interactive explorer.** Choose the residual (activity or abundance, three methods) in the
dropdown. A scatter of the two MAVE scores, coloured by residual, is linked to the heatmap:
hovering a point outlines its heatmap cell and vice versa (black). Clicking a point or cell
selects its residue (green): boxed in the heatmap, its variants ringed in the scatter and shown
as sticks on the AlphaFold structure (`AF-O60260-*.pdb`, py3Dmol); hovering the structure shows
residue labels. The structure is coloured by domain or by per-residue median residual (toggle):
it starts with domains, switches to residuals when a residue is selected, and "Reset markings"
clears all marks and returns to domain colours.

Code: data loading and per-residual quantities in `parkin_data.py`, figures in
`parkin_plots.py`, the linked explorer in `parkin_interactive.py` and the structure view in
`parkin_structure.py`. Quick checks of the explorer: `python tests/test_explorer.py`. The explorer needs the ipympl backend
(`%matplotlib widget`) and a running kernel; on Colab the structure viewer loads 3Dmol.js
from the web.


## Setup for Google Colab

This is the Colab version of `parkin_mave_comparison.ipynb` (the analysis cells are the same),
meant to be opened from the "Open in Colab" link in the repository README. Run the setup cell
first. It
1. installs the two packages Colab lacks: `ipympl` (interactive matplotlib) and `py3Dmol`
   (structure viewer), pinned to the versions used locally;
2. enables Colab's custom widget manager (needed for ipympl figures);
3. downloads only the files the notebook needs from the GitHub repository (`RAW_BASE`) into
   `/content/parkin_notebook`: the four `parkin_*.py` modules, `comparison_res.csv` and the
   AlphaFold structure (the viewer library 3Dmol.js is loaded from the web). Files already downloaded are kept.

If the interactive figures show only as text (e.g. `Canvas(...)`) after the setup, use
*Runtime → Restart session* and run all cells again (packages and files are kept).

In [ ]:
# ── Colab setup: packages, widget manager, project files from GitHub ──────────────
import os, sys, subprocess, urllib.request
from pathlib import Path

RAW_BASE = "https://raw.githubusercontent.com/enoee/parkin_comparison/main/colab/"   # repository branch (+ folder)
FILES = ["parkin_data.py", "parkin_plots.py", "parkin_interactive.py", "parkin_structure.py",
         "comparison_res.csv", "AF-O60260-F1-model_v2.pdb"]
WORK_DIR = Path("/content/parkin_notebook")

IN_COLAB = "google.colab" in sys.modules


def download_files(base_url, files, dest):
    """Download each file (relative path) from base_url into dest, skipping existing ones."""
    for name in files:
        target = Path(dest) / name
        if target.exists():
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", name)
        urllib.request.urlretrieve(base_url + name, target)


if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ipympl==0.10.0", "py3Dmol==2.5.4"], check=True)
    from google.colab import output
    output.enable_custom_widget_manager()
    download_files(RAW_BASE, FILES, WORK_DIR)
    BASE_DIR = WORK_DIR
else:
    BASE_DIR = Path(".")              # running locally: the project folder itself

missing = [f for f in FILES if not (BASE_DIR / f).exists()]
assert not missing, f"missing in {BASE_DIR.resolve()}: {missing}"
os.chdir(BASE_DIR)
sys.path.insert(0, str(BASE_DIR.resolve()))
print(f"Colab: {IN_COLAB} | project files in {BASE_DIR.resolve()}")

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import parkin_data as pdata
import parkin_plots as pp
from parkin_interactive import LinkedExplorer

%matplotlib widget
plt.rcParams.update({"font.family": "sans-serif", "pdf.fonttype": 42, "svg.fonttype": "none"})

In [ ]:
DATA_CSV = "comparison_res.csv"
PDB_FILE = sorted(glob.glob("AF-O60260-*.pdb"))[0]   # AlphaFold model for the 3D view

data = pdata.load_data(DATA_CSV)          # per-variant columns
annot = pdata.residue_table(data)         # per-residue wt, SS, domain, zf
print(f"{len(data['resi'])} variants over {len(annot['resi'])} residues")

## Annotation: domains, secondary structure and zinc sites

In [ ]:
zinc_sites = pdata.zinc_sites_from_column(annot["zf"])

print("Domains:", pdata.domain_segments(annot["domain"]))
print("Zn sites:", zinc_sites)

# secondary-structure parameters: DSSP codes and drawing style
SS_PARAMS = dict(helix_codes="HGI", strand_codes="E", min_helix=3, min_strand=2,
                 helix_period=3.6, helix_amp=0.32)

## Figure 1: residual heatmaps (A4 width, PDF)

Saves the activity and abundance PC spline residual heatmaps to `figures/`. Any other residual
can be saved with the button in the explorer below.

In [ ]:
os.makedirs("figures", exist_ok=True)
for residual in ["Activity: PC spline", "Abundance: PC spline"]:
    rd = pdata.residual_data(data, residual)   # residuals, scores, matrix, medians
    fig, _ = pp.heatmap_figure(rd, annot, zinc_sites, ss_kwargs=SS_PARAMS, pyplot=False)
    kind, method = pdata.RESIDUALS[residual]
    fig.savefig(f"figures/heatmap_{kind}_{method}.pdf")
    print("saved", f"figures/heatmap_{kind}_{method}.pdf")

## Interactive explorer
Hover over scatter points or heatmap cells; click one to select its residue (or type a residue
number; 0 = none). The structure keeps its orientation when the selection changes.

In [ ]:
explorer = LinkedExplorer(data, annot, zinc_sites, residual="Activity: PC spline",
                          ss_kwargs=SS_PARAMS, pdb_path=PDB_FILE, struct_vlim=0.3)
explorer.widget